# 04 — Agent Simulation

**Shin-Carley Framework — Approach A (Differentiated Python + Local LLM)**

## Formula Reference

### Fatigue — Three Process Model (Åkerstedt)
- `S_t = ha - (ha - S_w) * e^(d*taw)` &nbsp;&nbsp; ha=14.3, d=-0.0353
- `C = Ca * cos(2pi*(tod - p) / 24)` &nbsp;&nbsp; Ca=2.5, p=16.8 h
- `KSS = 10.6 - 0.6*(S + C + U)` &nbsp;&nbsp; [1=very alert, 9=very sleepy]
- `TotalFatigue = (KSS_norm + ED) / 2` (normalised to 1-5)

### Energy Depletion (Tian et al., 2022)
- `ED = 0.65*JobComplexity - 0.20*PsychEmpowerment + 1.80`
- Gender **removed** from formula — not a valid physiological predictor in this context

### Job Performance
- `JP1 = 2.766 - 0.106*Burnout + 0.301*IntrMotiv + 0.298*JobSat - 0.153*RoleConflict - 0.076*LeaveInt` (Rehman 2015)
- `JP2 = 3.238 - 0.022*TimePressure - 0.086*Workload - 0.141*LackMotiv - 0.155*RoleAmb` (Basit & Hassan 2017)
- `JP_final = (JP1 + JP2) / 2 - 0.34*TotalFatigue` (crossover penalty: Hassan & Morsy -2.442/pt scaled)

### Flawed Perception Level (Shonman / ACT-R / IBLT)
- `FPL = 0.5 * fatigue_norm * (1 - jp_norm)` &nbsp;&nbsp; [0.0-0.5]
- 0.0 = perfect observer; 0.5 = random guessing

## Pipeline
1. Cues extracted per email (Ollama/Gemini/regex, cached to `data/cue_cache/`)
2. N agents with random traits; fatigue is **time-varying** via KSS (not static)
3. Each agent processes every email at 5 workday hours
4. Per-cue FPL governs stochastic cue-missing: clicked or reported
5. Results analysed: click rate, fatigue effect, trait correlations, cue heatmap

## 0 — Setup

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')

from src.agent import Agent
from src.cue_extractor import CueExtractor
from src.ollama_extractor import OllamaExtractor
from src.decision_loop import simulate_email
from src.simulation import run_simulation, save_results, click_rate_summary

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

print('All imports OK')
print(f'Root: {ROOT}')

In [ ]:
# ── Ollama availability check ───────────────────────────────────────────────
# Run: ollama pull qwen2.5:7b   (or gemma2:9b) before using
_oe = OllamaExtractor(model='qwen2.5:7b')
if _oe.is_available():
    print('Ollama OK — qwen2.5:7b is ready.')
    print('Set USE_OLLAMA=True in the simulation cell to use local LLM extraction.')
    print('Remember to delete data/cue_cache/ first for full re-extraction.')
else:
    print('Ollama not detected. Will use cached Gemini/regex extractions.')
    print('To enable: install Ollama, then run: ollama pull qwen2.5:7b')

## 1 — Spot-check: single agent, single email

In [ ]:
agent = Agent.random_agent('demo_agent', seed=7)

print('=== Agent traits ===')
print(f'  age={agent.age:.1f}, education={agent.education_level:.1f}, tenure={agent.tenure:.1f}y')
print(f'  job_type={int(agent.job_type)} (1=desk), job_complexity={agent.job_complexity:.1f}')
print(f'  intrinsic_motivation={agent.intrinsic_motivation:.1f}  (psych. empowerment proxy in ED)')
print(f'  suspicion_threshold={agent.suspicion_threshold}, '
      f'max_cues_processed={agent.max_cues_processed}')
print(f'  NOTE: gender={int(agent.gender)} stored for demographics but NOT used in any formula')
print()

print('=== Three Process Model state across workday ===')
header = f'{"Hour":>6}  {"KSS":>6}  {"TotalFatigue":>12}  {"FinalJP":>8}  {"FPL":>6}  {"TimePressure":>12}'
print(header)
print('-' * 65)
for hour in [8.0, 10.0, 12.0, 14.0, 16.0]:
    agent.advance_workday(hour)
    kss = agent.compute_kss()
    tf  = agent.compute_total_fatigue()
    jp  = agent.compute_job_performance()
    fpl = agent.compute_flawed_perception_level()
    print(f'{int(hour):>4}:00  {kss:>6.2f}  {tf:>12.2f}  {jp:>8.2f}  {fpl:>6.3f}  {agent.time_pressure:>12.1f}')

print()
print('Key: KSS decreases as circadian alertness rises through the day.')
print('FPL rises later because time_pressure/workload degrade JP faster than KSS improves.')

In [ ]:
# Decision loop demo — same email, same agent, at 8am vs 4pm
sample_cues = ['urgency', 'generic_greeting', 'personal_info', 'suspicious_link', 'threats']

print('=== Decision loop demo ===')
for hour in [8.0, 12.0, 16.0]:
    agent.advance_workday(hour)
    result = simulate_email(agent, sample_cues)
    print(f'  {int(hour):02d}:00 fpl={result["fpl"]:.3f} → '
          f'{result["decision"].upper():8s} '
          f'(perceived {result["suspicion_counter"]}/{len(sample_cues)} cues: '
          f'{result["cues_perceived"]})')

In [ ]:
# ── KSS trajectory across workday — Three Process Model in action ──────────
import numpy as np

hours = np.arange(8.0, 17.5, 0.5)

# Three archetypes to show variance
archetype_specs = [
    ('Well-rested (sleep=8h, quality=5)', 5.0, 8.0, 7),
    ('Average (sleep=7.5h, quality=3)',   3.0, 7.5, 7),
    ('Sleep-deprived (sleep=5h, quality=1)', 1.0, 5.0, 7),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, sq, st, seed in archetype_specs:
    ag = Agent.random_agent(f'arch_{seed}', seed=seed)
    ag.sleep_quality    = sq
    ag.total_sleep_time = st
    kss_vals, fpl_vals = [], []
    for h in hours:
        ag.advance_workday(float(h))
        kss_vals.append(ag.compute_kss())
        fpl_vals.append(ag.compute_flawed_perception_level())
    axes[0].plot(hours, kss_vals, marker='.', label=label)
    axes[1].plot(hours, fpl_vals, marker='.', label=label)

axes[0].set_title('KSS (Karolinska Sleepiness Scale)\n1=very alert, 9=very sleepy')
axes[0].set_xlabel('Workday hour')
axes[0].set_ylabel('KSS score')
axes[0].set_ylim(1, 9)
axes[0].set_xticks([8, 10, 12, 14, 16])
axes[0].set_xticklabels(['8am','10am','12pm','2pm','4pm'])
axes[0].axhspan(7, 9, alpha=0.08, color='red', label='High sleepiness zone')
axes[0].legend(fontsize=8)

axes[1].set_title('Flawed Perception Level (FPL)\nHigher = more phishing cues missed')
axes[1].set_xlabel('Workday hour')
axes[1].set_ylabel('FPL')
axes[1].set_ylim(0, 0.5)
axes[1].set_xticks([8, 10, 12, 14, 16])
axes[1].set_xticklabels(['8am','10am','12pm','2pm','4pm'])
axes[1].axhspan(0.3, 0.5, alpha=0.08, color='red', label='High-risk FPL zone')
axes[1].legend(fontsize=8)

plt.suptitle('Three Process Model: time-varying fatigue drives FPL (Åkerstedt + Shin-Carley)', fontsize=11)
plt.tight_layout()
plt.show()

## 2 — Run the full simulation

> **IMPORTANT: agent formulas updated.**  
> The previous `simulation_results.csv` used static fatigue (no TPM) and `gender` in the ED formula.  
> `RERUN = True` below — re-run before analysing results.
>
> **Cue extraction options** (set `USE_OLLAMA`):
> - `True` — uses local Ollama model (richer cue extraction, especially for phishbowl emails)
> - `False` — uses cached Gemini/regex results (fast, no API needed)

In [ ]:
results_path = ROOT / 'data' / 'simulation_results.csv'

# ─── FLAGS ────────────────────────────────────────────────────────────────────
RERUN      = True   # re-run after formula changes (TPM fatigue, gender removed from ED)
USE_OLLAMA = False  # True = use local Ollama for extraction (delete cue_cache/ first)
# ─────────────────────────────────────────────────────────────────────────────

if USE_OLLAMA:
    import src.simulation as _sim_mod
    _sim_mod.CueExtractor = OllamaExtractor
    print('Cue extraction: OllamaExtractor (qwen2.5:7b)')
else:
    print('Cue extraction: CueExtractor (Gemini/regex cache)')

if not results_path.exists() or RERUN:
    df = run_simulation(
        emails_csv=str(ROOT / 'data' / 'processed' / 'master_emails.csv'),
        n_agents=30,
        workday_hours=[8.0, 10.0, 12.0, 14.0, 16.0],
        seed=42,
        cache_dir=str(ROOT / 'data' / 'cue_cache'),
    )
    save_results(df, str(results_path))
else:
    df = pd.read_csv(results_path)
    print(f'Loaded existing results: {len(df):,} rows')

df.head()

## 3 — Validate cue extraction

In [ ]:
# How many cues did Gemini find per email source?
cue_stats = (
    df[['email_id', 'source', 'actual_class', 'cues_extracted']]
    .drop_duplicates('email_id')
    .groupby('source')['cues_extracted']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .round(2)
)
print('Cues extracted per email by source:')
print(cue_stats)

fig, ax = plt.subplots(figsize=(8, 4))
per_email = df[['email_id', 'source', 'cues_extracted']].drop_duplicates('email_id')
sns.boxplot(data=per_email, x='source', y='cues_extracted', ax=ax)
ax.set_title('Cues extracted by Gemini — per email source')
ax.set_xlabel('')
ax.set_ylabel('Cue count')
plt.tight_layout()
plt.show()

## 4 — Click rates: phishing emails

In [ ]:
# Click rate = fraction of phishing emails where agent chose 'clicked' (vulnerability)
phishing_df = df[df['actual_class'] == 1].copy()
phishing_df['clicked'] = (phishing_df['decision'] == 'clicked').astype(int)

by_source_hour = (
    phishing_df.groupby(['source', 'workday_hour'])['clicked']
    .mean()
    .reset_index()
    .rename(columns={'clicked': 'click_rate'})
)

fig, ax = plt.subplots(figsize=(9, 5))
for source, grp in by_source_hour.groupby('source'):
    ax.plot(grp['workday_hour'], grp['click_rate'], marker='o', label=source)
ax.set_xlabel('Workday hour')
ax.set_ylabel('Click rate (phishing)')
ax.set_title('Agent phishing click rate across the workday by email source')
ax.legend()
ax.set_xticks([8, 10, 12, 14, 16])
ax.set_xticklabels(['8am', '10am', '12pm', '2pm', '4pm'])
plt.tight_layout()
plt.show()

In [ ]:
# False positive rate: how often do agents 'report' benign emails?
benign_df = df[df['actual_class'] == 0].copy()
benign_df['reported'] = (benign_df['decision'] == 'reported').astype(int)

fpr_by_hour = (
    benign_df.groupby('workday_hour')['reported']
    .mean()
    .reset_index()
    .rename(columns={'reported': 'false_positive_rate'})
)
print('False positive rate (benign emails incorrectly reported as phishing):')
print(fpr_by_hour.to_string(index=False))

## 5 — Fatigue effect on click rate

In [ ]:
# Bin agents into low/medium/high fatigue quartiles
phishing_df['fatigue_bin'] = pd.qcut(
    phishing_df['total_fatigue'], q=3, labels=['Low', 'Medium', 'High']
)

fatigue_click = (
    phishing_df.groupby(['fatigue_bin', 'workday_hour'])['clicked']
    .mean()
    .reset_index()
    .rename(columns={'clicked': 'click_rate'})
)

fig, ax = plt.subplots(figsize=(9, 5))
for fb, grp in fatigue_click.groupby('fatigue_bin', observed=True):
    ax.plot(grp['workday_hour'], grp['click_rate'], marker='o', label=f'Fatigue: {fb}')
ax.set_xlabel('Workday hour')
ax.set_ylabel('Click rate')
ax.set_title('Phishing click rate by fatigue level across the workday')
ax.legend()
ax.set_xticks([8, 10, 12, 14, 16])
ax.set_xticklabels(['8am', '10am', '12pm', '2pm', '4pm'])
plt.tight_layout()
plt.show()

## 6 — Agent trait correlations

In [ ]:
# Per-agent click rate on phishing emails (averaged across all hours and emails)
agent_summary = (
    phishing_df.groupby('agent_id')
    .agg(
        click_rate=('clicked', 'mean'),
        avg_fatigue=('total_fatigue', 'mean'),
        avg_jp=('final_jp', 'mean'),
        avg_fpl=('fpl', 'mean'),
        age=('age', 'first'),
        education_level=('education_level', 'first'),
        job_complexity=('job_complexity', 'first'),
        suspicion_threshold=('suspicion_threshold', 'first'),
    )
    .reset_index()
)

# Correlation heatmap
corr_cols = ['click_rate', 'avg_fatigue', 'avg_jp', 'avg_fpl',
             'age', 'education_level', 'job_complexity', 'suspicion_threshold']
corr = agent_summary[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation: agent traits vs phishing click rate')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: FPL vs click rate
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(agent_summary['avg_fpl'], agent_summary['click_rate'], alpha=0.7)
axes[0].set_xlabel('Average FPL (flawed perception level)')
axes[0].set_ylabel('Click rate on phishing')
axes[0].set_title('FPL vs click rate')

axes[1].scatter(agent_summary['suspicion_threshold'], agent_summary['click_rate'],
                alpha=0.7, color='coral')
axes[1].set_xlabel('Suspicion threshold')
axes[1].set_ylabel('Click rate on phishing')
axes[1].set_title('Suspicion threshold vs click rate')

plt.tight_layout()
plt.show()

## 7 — Cue heatmap (which cues drive reporting)

In [ ]:
import ast

# Expand cues_perceived into one-hot columns
all_cues = [
    'urgency', 'threats', 'generic_greeting', 'spelling_grammar',
    'emotional_appeal', 'too_good_true', 'personal_info',
    'suspicious_sender', 'suspicious_link'
]

def parse_cues(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

phishing_df = phishing_df.copy()
phishing_df['cues_perceived'] = phishing_df['cues_perceived'].apply(parse_cues)

for cue in all_cues:
    phishing_df[f'perceived_{cue}'] = phishing_df['cues_perceived'].apply(
        lambda c: int(cue in c)
    )

# How often each cue is perceived per source
perceived_cols = [f'perceived_{c}' for c in all_cues]
cue_by_source = (
    phishing_df.groupby('source')[perceived_cols].mean() * 100
).round(1)
cue_by_source.columns = all_cues

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(cue_by_source, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set_title('% of trials where cue was perceived — by source')
ax.set_ylabel('Email source')
ax.set_xlabel('Cue')
plt.tight_layout()
plt.show()

## 8 — Summary table

In [ ]:
summary = click_rate_summary(df)

# Phishing click rate (vulnerability) by source
print('Phishing click rate (vulnerability) — lower is better:')
print(summary[summary['actual_class'] == 1]
      .pivot(index='source', columns='workday_hour', values='click_rate')
      .to_string())

print()
print('Benign click rate (correct) — higher is better:')
print(summary[summary['actual_class'] == 0]
      .pivot(index='source', columns='workday_hour', values='click_rate')
      .to_string())